# 4.16 · 等张回归 / Isotonic Regression —— Part 4 收官 🏁

> **课程定位 / Where this fits**
> **Part 4 最后一课**。前 15 课要么假设参数形式（线性/多项式/指数）, 要么是黑盒（树）。等张回归只假设**一件事: 单调**（y 随 x 不减），其余完全由数据决定——无参数、无函数形式。它的杀手应用是**概率校准**（Part 5.8）和剂量-反应曲线。
> Isotonic regression assumes only monotonicity (y non-decreasing in x), nothing else. Its killer app is probability calibration.

> 💡 **面试相关 / Interview-relevant**
> - "什么是等张回归" ★★★（单调约束的非参数拟合）
> - "模型概率不准怎么校准" ★★★★（isotonic / Platt, 接 Part 5.8）
> - "PAV 算法" ★★（相邻违反者合并）

---

## 学习目标 / Learning Objectives
1. 理解**单调约束**回归: 唯一假设是 y 不减。
2. 理解 **PAV (Pool Adjacent Violators)** 算法。
3. 等张 vs 线性 vs 多项式: 灵活性 vs 约束。
4. **杀手应用: 概率校准**（接 Part 5.8 模型校准）。

## 目录 / TOC
1. [单调约束: 唯一的假设 ⭐](#1)
2. [PAV 算法 ⭐](#2)
3. [等张 vs 其他回归](#3)
4. [杀手应用: 概率校准 ⭐](#4)
5. [小结 + Part 4 总结 🏁](#5)


<a id="1"></a>
## 1. 单调约束: 唯一的假设 ⭐ / The Only Assumption

等张回归（isotonic = 保序）拟合一个**单调不减**的阶梯函数, 在此约束下最小化平方误差：

$$\min_{\hat{y}} \sum_i (y_i - \hat{y}_i)^2 \quad \text{s.t.} \quad \hat{y}_1 \le \hat{y}_2 \le \dots \le \hat{y}_n \;(\text{当 } x \text{ 升序})$$

**和前面所有模型的根本区别**:
- 线性/多项式: 假设**具体函数形式**（直线/曲线）
- 等张: **不假设任何形式**, 只要求"单调"——形状完全由数据决定

**何时用**: 你**有领域知识确信关系单调**（剂量越大反应越强、价格越高需求越低、模型分数越高真实概率越高）, 但**不知道具体形状**。单调约束既给了灵活性, 又防止过拟合（比自由非参数稳）。
Use it when you know the relationship is monotonic but not its shape — flexible yet regularized by monotonicity.


In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.isotonic import IsotonicRegression
sns.set_theme(style="whitegrid")
rng = np.random.default_rng(42)

# 单调但形状未知的关系 + 噪声 / monotonic but unknown shape
n = 80
x = np.sort(rng.uniform(0, 10, n))
y_true = 5 / (1 + np.exp(-(x - 5)))      # 真实是 S 形 (单调)
y = y_true + rng.normal(0, 0.5, n)

iso = IsotonicRegression(out_of_bounds="clip").fit(x, y)
fig, ax = plt.subplots(figsize=(7, 4.5))
ax.scatter(x, y, alpha=0.4, s=20, label="数据(单调+噪声)")
ax.plot(x, y_true, "g--", lw=1.5, label="真实(S形)")
ax.plot(x, iso.predict(x), "r-", lw=2, label="等张回归(单调阶梯)")
ax.legend(); ax.set_title("等张回归: 只假设单调, 拟合出单调阶梯函数\n无需知道是S形/线性/任何形式")
plt.tight_layout(); plt.show()
print("等张回归拟合出单调非减的阶梯, 自动贴合 S 形 — 完全没告诉它是 S 形")


<a id="2"></a>
## 2. PAV 算法 ⭐ / Pool Adjacent Violators

等张回归用 **PAV (Pool Adjacent Violators)** 算法精确求解, $O(n)$：

```
1. 从左到右扫描
2. 若发现"违反者"(后一个 < 前一个, 违反单调) →
   把它们"合并(pool)"成一个块, 块值 = 块内平均
3. 合并可能引发新违反 → 继续向左合并, 直到全局单调
```

**直觉**: 哪里数据违反了单调, 就把那一段**抹平成平均值**, 直到整体单调。结果是分段常数（阶梯）。
PAV: wherever data violates monotonicity, pool the violators into their average until the whole sequence is monotone.


In [ ]:
# 从零实现 PAV / from-scratch PAV
def pav(y):
    y = y.astype(float).copy()
    n = len(y)
    # 用块表示: (加权和, 权重, 值) / blocks
    vals = list(y); weights = [1.0]*n
    i = 0
    while i < len(vals) - 1:
        if vals[i] > vals[i+1]:           # 违反单调 / violation
            # 合并 i 和 i+1 / pool them
            new_w = weights[i] + weights[i+1]
            new_v = (vals[i]*weights[i] + vals[i+1]*weights[i+1]) / new_w
            vals[i] = new_v; weights[i] = new_w
            del vals[i+1]; del weights[i+1]
            if i > 0: i -= 1              # 回退检查新违反 / back up
        else:
            i += 1
    # 展开成原长度 / expand back
    result = []
    for v, w in zip(vals, weights):
        result.extend([v]*int(w))
    return np.array(result)

# 对照 sklearn / verify against sklearn
my_pav = pav(y)
sk_iso = IsotonicRegression().fit_transform(x, y)
print(f"从零 PAV vs sklearn 最大差异: {np.abs(my_pav - sk_iso).max():.6f} → 一致")
print(f"输出块数(阶梯段数): {len(np.unique(np.round(my_pav, 6)))} (原 {n} 点被合并成几段)")
print("PAV 把违反单调的相邻点合并成平均, 直到全局单调 — O(n) 精确解")


<a id="3"></a>
## 3. 等张 vs 其他回归 / Comparison


In [ ]:
from sklearn.linear_model import LinearRegression
from sklearn.preprocessing import PolynomialFeatures
from sklearn.pipeline import make_pipeline

fig, ax = plt.subplots(figsize=(8, 4.5))
ax.scatter(x, y, alpha=0.35, s=20, label="数据")
xp = x.reshape(-1, 1)
ax.plot(x, LinearRegression().fit(xp, y).predict(xp), label="线性(假设直线)")
ax.plot(x, make_pipeline(PolynomialFeatures(3), LinearRegression()).fit(xp, y).predict(xp), label="3阶多项式")
ax.plot(x, IsotonicRegression(out_of_bounds="clip").fit(x, y).predict(x), "r-", lw=2.5, label="等张(只假设单调)")
ax.legend(fontsize=9); ax.set_title("线性(太死板) vs 多项式(可能非单调) vs 等张(灵活且保证单调)")
plt.tight_layout(); plt.show()
print("线性: 太死板, 假设直线; 多项式: 灵活但可能违反单调(尾部翘起);")
print("等张: 灵活贴合数据 + 保证单调 — 当你确信单调但不知形状时最佳")


**等张回归的位置**：
| 模型 | 假设 | 灵活性 | 风险 |
|---|---|---|---|
| 线性 | 直线 | 低 | 欠拟合非线性 |
| 多项式 | 多项式曲线 | 中 | 可能非单调/过拟合 |
| 等张 | **仅单调** | 高 | 阶梯不平滑; 外推靠 clip |
| 自由非参(KNN/样条) | 几乎无 | 最高 | 易过拟合 |

等张的"单调约束"是一种**恰到好处的正则化**: 比自由非参稳, 比参数模型灵活。


<a id="4"></a>
## 4. 杀手应用: 概率校准 ⭐ / Killer App: Probability Calibration

**等张回归最重要的工业用途**（接 Part 5.8）。很多分类器输出的"概率"**不准**——比如它说"0.9 概率是正类"的那些样本里, 实际只有 70% 是正类。模型**排序对**（AUC 好）但**概率值偏**。

**校准**: 学一个映射, 把模型的原始分数变成**真实概率**。等张回归天然适合——它**单调**（保持排序不变 = AUC 不变）, 又能任意弯曲去对齐真实概率。
- **Isotonic 校准**: 等张回归（灵活, 需较多数据）
- **Platt 校准**: sigmoid 拟合（参数化, 小数据更稳）

下面演示一个"过度自信"的模型如何被等张校准修正。
Isotonic calibration maps raw scores to true probabilities, preserving ranking (monotone) while bending to match reality.


In [ ]:
from sklearn.calibration import calibration_curve
from sklearn.linear_model import LogisticRegression

# 造一个"过度自信"的分数 / an overconfident score
n2 = 2000
true_p = rng.uniform(0, 1, n2)
labels = (rng.uniform(0, 1, n2) < true_p).astype(int)   # 真实标签按 true_p
# 模型分数: 排序对但过度自信(把概率推向0/1) / overconfident scores
raw_score = np.clip(true_p**0.4 if False else 1/(1+np.exp(-6*(true_p-0.5))), 0, 1)

# 等张校准 / isotonic calibration
iso_cal = IsotonicRegression(out_of_bounds="clip").fit(raw_score, labels)
calibrated = iso_cal.predict(raw_score)

# 校准曲线 / reliability diagram
fig, ax = plt.subplots(figsize=(6, 5.5))
ax.plot([0,1],[0,1],"k--",label="完美校准")
for scores, name, c in [(raw_score,"原始分数(过度自信)","C3"), (calibrated,"等张校准后","C2")]:
    frac_pos, mean_pred = calibration_curve(labels, scores, n_bins=10, strategy="quantile")
    ax.plot(mean_pred, frac_pos, "o-", color=c, label=name)
ax.set_xlabel("预测概率"); ax.set_ylabel("实际正类比例"); ax.legend()
ax.set_title("概率校准: 原始分数偏离对角线(不准)\n等张校准后贴合对角线(概率可信)")
plt.tight_layout(); plt.show()
print("原始分数: 校准曲线偏离对角线 (说 0.9 实际没那么高) = 过度自信")
print("等张校准后: 贴合对角线 → '0.9 的预测里真有~90% 是正类' = 概率可信")
print("\n💡 等张/Platt 校准是 Part 5.8 的正题; 这里展示等张回归的最重要工业用途")


**为什么校准重要**: 很多决策依赖**概率值**而非仅排序——风控阈值、医疗诊断置信度、期望损失计算（2.10/2.12）。一个 AUC 0.95 但概率不准的模型, 在"按 0.8 概率拒贷"这种决策上会出错。等张回归用单调约束**修正概率值而不破坏排序**, 是校准的标准工具。
Calibration matters when decisions use probability values, not just ranking — isotonic fixes the values without breaking the order.


<a id="5"></a>
## 5. 小结 + Part 4 总结 🏁 / Summary

### 等张回归小结
```
等张回归: 唯一假设=单调(y随x不减), 无函数形式, 数据定形状
PAV 算法: 相邻违反者合并成平均, O(n) 精确, 输出阶梯
定位: 比参数模型灵活, 比自由非参稳(单调=正则化)
杀手应用 ⭐: 概率校准(Part 5.8) — 单调保排序(AUC不变)+弯曲对齐真实概率
```

---

## 🏁 Part 4 全部完成 / Part 4 Complete!

| # | 模型 | 一句话核心 |
|---|---|---|
| 4.1 | 线性回归 | 正规方程 + 三视角 + 五假设 |
| 4.2 | 回归诊断 | 残差图 + 异方差 + VIF + 影响点 |
| 4.3 | 多项式 | 基函数展开 + 偏差-方差可视化 |
| 4.4 | Ridge | L2 收缩 + 治共线 + =高斯先验 |
| 4.5 | Lasso | L1 稀疏 + 特征选择 + =Laplace先验 |
| 4.6 | Elastic Net | L1+L2 + 分组效应 |
| 4.7 | GLM | link function 统一指数族 |
| 4.8 | 非线性回归 | curve_fit + 初值 + 逻辑增长 |
| 4.9 | SVR | ε-不敏感 + 核技巧 |
| 4.10 | KNN | 惰性 + 维度诅咒 |
| 4.11 | 决策树 | 递归切分 + 不需缩放 + 高方差 |
| 4.12 | 随机森林 | bagging 降方差 + OOB |
| 4.13 | GBDT/XGBoost | boosting 降偏差 + 拟合残差 |
| 4.14 | 分位数回归 | pinball loss + 预测区间 |
| 4.15 | 稳健回归 | Huber/RANSAC/Theil-Sen 抗异常 |
| 4.16 | 等张回归 | 单调约束 + 概率校准 |

**贯穿 Part 4 的主线**：
1. **损失函数决定模型性格**（2.9 的"损失=噪声模型"在此处处印证）：平方→均值, 绝对/pinball→中位数/分位数, ε-不敏感→SVR, Huber→稳健, 残差梯度→boosting
2. **正则化 = 先验**（4.4/4.5 ↔ 2.10）：L2=高斯, L1=Laplace
3. **偏差-方差权衡贯穿始终**：多项式阶数、Ridge λ、KNN k、树深度、RF 树数、GBDT 迭代——全是同一个旋钮的不同形态
4. **树家族统治表格数据**：单树(弱)→森林(降方差)/GBDT(降偏差)→XGBoost(工程巅峰)

### 下一站
**Part 5 · 监督学习：分类**——回归预测连续值, 分类预测类别。逻辑回归、SVM、朴素贝叶斯、XGBoost 分类、评估指标(ROC/PR/F1)、不平衡、校准。很多概念(正则/树/boosting)直接迁移。
